# Fine-tune TECSAS subcompartments for a new cell type (50 kb / hg19)

This notebook adapts the pre-trained **GM12878** TECSAS model to predict the 5 chromatin
subcompartment classes (**A1, A2, B1, B2, B3**) for a *different* cell type, using **your own**
Hi-C-derived subcompartment labels for training.

**Approach (transfer learning):** the encoder + transformer of TECSAS operate per sequence
position and are independent of how many epigenomic experiments a cell type has. We therefore
reuse the pre-trained GM12878 encoder as a *frozen* feature extractor and train only a fresh
output head (`l2`) sized to the target cell line's data. This is the same recipe used in
`train_and_predict_XADS_HistMod_RNASeq.ipynb`, kept on the original 5-class subcompartment task.

All of the heavy lifting lives in [`finetune_subcompartments.py`](finetune_subcompartments.py);
this notebook just configures and runs it.

## 0. Requirements

- Network access to ENCODE (to download the target cell line's tracks) and a GPU for training.
- The TECSAS package importable (`pip install -e .` from the repo root, or `pip install TECSAS`).
- **Your own subcompartment labels** for a training cell line (see the format below).

In [ ]:
import os
os.system('pip install -q pyBigWig requests joblib tqdm urllib3')
# If running from a fresh checkout in Colab, uncomment:
# !git clone https://github.com/ed29rice/TECSAS.git && pip install -e TECSAS

## 1. Provide your labels

Create a directory (`labels_dir`) with **one file per chromosome**, named
`chr1_beads.txt.original` ... `chr22_beads.txt.original`. Each file is space-delimited with two
columns, `<bin_index> <label>`:

```
1 NA
2 A1
3 A1
4 B1
...
```

- The **label is in column index 1**, one of `A1 A2 B1 B2 B3` (`NA` is allowed and is excluded
  from training; `B4` is also excluded).
- One row **per 50 kb bin**, in order. The number of rows for each chromosome must match
  `data_process.chrm_size` (hg19 @ 50 kb).
- The shipped GM12878 labels in `TECSAS/share/subcom_GM12878_50kb/` are a concrete example of
  this exact format.

In [ ]:
from Tutorials.finetune_subcompartments import Config, run_finetune

cfg = Config(
    cell_line='MyCell',                 # <-- ENCODE-recognised cell-line name (auto-downloaded)
    assembly='hg19',
    res=50,                             # keep 50 kb to match the shipped model
    labels_dir='./my_labels_50kb',      # <-- YOUR chr*_beads.txt.original files
    histones=True, tf=True,             # assays to pull from ENCODE
    pretrained='TECSAS/share/models/bv_GM12878_155.pt',
    train_chroms=list(range(2, 23, 2)), # even chromosomes for train/val
    test_chroms=list(range(1, 23, 2)),  # odd chromosomes held out for evaluation
    epochs=75,
    output_dir='./finetune_MyCell',
    nproc=10,
)
cfg

## 2. Run the fine-tune

This downloads + processes the target cell line's ENCODE tracks and the GM12878 reference,
screens the experiments (`filter_exp`), assembles the train/val/test matrices, loads the frozen
GM12878 encoder, trains a fresh head, and writes predictions.

**Tip:** set `Config(..., download=False)` on later runs to reuse already-processed tracks and
skip the (slow) ENCODE download.

In [ ]:
txt_path, bed_path, accuracy = run_finetune(cfg)
print('Held-out accuracy:', accuracy)
print('Predictions:', txt_path)
print('BED track:', bed_path)

## 3. Outputs

In `output_dir` you will find:

- `best_val_model_params_<cell_line>.pt` — the fine-tuned weights (best validation loss).
- `subcompartments_<cell_line>_predictions.txt` — integer labels (`0=A1, 1=A2, 2=B1, 3=B2, 4=B3`).
- `subcompartments_<cell_line>.bed` — a colored BED track you can load in a genome browser.

You can also inspect the per-class confusion matrix to see where the fine-tuned model agrees
with your Hi-C-derived labels on the held-out chromosomes:

In [ ]:
import numpy as np
pred = np.loadtxt(txt_path, dtype=int)
print('Predicted class counts (0=A1..4=B3):', np.unique(pred, return_counts=True))